In [ ]:
%load_ext autoreload
%autoreload

In [36]:
# Import required libraries and functions
import numpy as np
import paseos
import pykep as pk
from dotmap import DotMap
import toml
from paseos import ActorBuilder, SpacecraftActor, GroundstationActor

# Import funtions from licos
import sys
import os
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)
from licos.get_constellation import get_constellation, get_constellation_with_tle


In [37]:
# Import cfg file as DotMap
path = "../cfg/old/simulation_without_training_cfg.toml"
with open(path) as cfg:
    # dynamic=False inhibits automatic generation of non-existing keys
    cfg = DotMap(toml.load(cfg), _dynamic=False)

In [38]:
# Define Constellation
nPlanes = 1             # the number of orbital planes
nSats = 8               # the number of satellites per orbital plane

# Define the orbit:
#   1. Using altitude and inclination: 
altitude = 786 * 1000   # altitude above the Earth's ground [m]
inclination = 98.62     # inclination of the orbit

#   2. Using TLE (accessed 2024-06-27 13:22:10 CET at https://www.n2yo.com/satellite/?s=40697)
line1 = "1 40697U 15028A   24179.17258119  .00000251  00000-0  11245-3 0  9993"
line2 = "2 40697  98.5685 253.3004 0001246  98.7583 261.3741 14.30825311470776"

# Select approach (with or without tle):
with_tle = True

# Starting date of our simulation
t0 = pk.epoch_from_string("2021-Mar-20 12:40:00")  

In [39]:
# Initialize simulation
cfg = paseos.load_default_cfg()  # loading cfg to modify defaults
cfg.sim.start_time = t0.mjd2000 * pk.DAY2SEC  # convert epoch to seconds
paseos_instances = []

# Create satellite constellation
if with_tle:
    planet_list, sats_pos_and_v, _ = get_constellation_with_tle(line1, line2, nSats, nPlanes, t0, verbose=False)
else:
    planet_list, sats_pos_and_v, _ = get_constellation(altitude, inclination, nSats, nPlanes, t0, verbose=False)


earth = pk.planet.jpl_lp("earth")  # define our central body

# Create Satellite actors and add to paseos simulation
for sat in range(nSats):
    pos, v = sats_pos_and_v[sat]  # get our position and velocity

    # Create the local actor, name will be the rank
    local_actor = ActorBuilder.get_actor_scaffold(
        name="Sat_" + str(sat), actor_type=SpacecraftActor, epoch=t0
    )
    ActorBuilder.set_orbit(
        actor=local_actor, position=pos, velocity=v, epoch=t0, central_body=earth
    )

    # Add devices to the local actor
    ActorBuilder.add_comm_device(
        actor=local_actor, device_name="Link1", bandwidth_in_kbps=1000)

    ActorBuilder.set_power_devices(
            actor=local_actor,
            battery_level_in_Ws=277200 * 0.5,
            max_battery_level_in_Ws=277200,
            charging_rate_in_W=20)

    ActorBuilder.set_thermal_model(
            actor=local_actor,
            actor_mass=6.0,
            actor_initial_temperature_in_K=283.15,
            actor_sun_absorptance=0.9,
            actor_infrared_absorptance=0.5,
            actor_sun_facing_area=0.012,
            actor_central_body_facing_area=0.01,
            actor_emissive_area=0.1,
            actor_thermal_capacity=6000)
    
    instance = paseos.init_sim(local_actor=local_actor, cfg=cfg)
    paseos_instances.append(instance)

# Add Ground stations
comms_instances = []
stations = [
    #["Maspalomas", 27.7629, -15.6338, 205.1],
    #["Matera", 40.6486, 16.7046, 536.9],
    ["Svalbard", 78.9067, 11.8883, 474.0],
    ["Disaster Site", -31.6468, 152.7993, 5]]

for station in stations:
    gs_actor = ActorBuilder.get_actor_scaffold(
        name=station[0], actor_type=GroundstationActor, epoch=t0
    )
    ActorBuilder.set_ground_station_location(
        gs_actor,
        latitude=station[1],
        longitude=station[2],
        elevation=station[3],
        minimum_altitude_angle=5,
    )
    instance = paseos.init_sim(local_actor=gs_actor)
    comms_instances.append(instance)

In [ ]:
n_passes = 0
passes = []
time_to_advance = 60 #Advance time by 1 minute at a time

# Simulate the constellation for ~24 hours from given initial epoch
for val in np.arange(0, 86400, time_to_advance):
    if n_passes == 6:
            # We end the sim after 2 consecutive passes with 
            # different satellites and LOS of the disaster site
            break
    
    for idx, instance in enumerate(paseos_instances):
        disaster_in_sight = instance.local_actor.is_in_line_of_sight(comms_instances[-1].local_actor, instance.local_time)
        if disaster_in_sight:
            time_of_pass = instance.local_time

            # If no passes have been detected yet, store and print the time of first pass with LOS of the disaster site
            if len(passes) == 0:
                passes.append((time_of_pass, idx))
                print("Pass", n_passes+1, 
                      "at time: ", instance.local_time, "  ", 
                      "(Sat", idx, "at site:",comms_instances[-1].local_actor.name, ")   ",
                      "Delta_t: 0")
                n_passes += 1

            # If two consecutive passes have been detected, print time of the second pass with LOS of the disaster site
            else:

                #If we consider multiple sats, ensure that the pass is from a different satellite
                if idx != passes[-1][1] and nSats > 1:

                    # Get time between first and second pass with LOS of the disaster site
                    delta_t = instance.local_time.mjd2000 * pk.DAY2SEC- passes[-1][0].mjd2000 * pk.DAY2SEC

                    # Print time of the second pass with LOS of the disaster site
                    print("Pass", n_passes+1, 
                          "at time: ", instance.local_time, "  ",
                          "(Sat", idx, "at site:",comms_instances[-1].local_actor.name, ")   ",
                          "Delta_t:", delta_t, "[s]  (or", delta_t/60, "[min])")
                    
                    # Save results
                    passes.append((time_of_pass, idx))
                    n_passes += 1

                # If we consider only one sat, ensure that it is a different pass
                elif nSats == 1 and (instance.local_time.mjd2000 * pk.DAY2SEC- passes[-1][0].mjd2000 * pk.DAY2SEC) > 1000:

                    # Get time between first and second pass with LOS of the disaster site
                    delta_t = instance.local_time.mjd2000 * pk.DAY2SEC- passes[-1][0].mjd2000 * pk.DAY2SEC

                    # Print time of the second pass with LOS of the disaster site
                    print("Pass", n_passes+1, 
                          "at time: ", instance.local_time, "  ",
                          "(Sat", idx, "at site:",comms_instances[-1].local_actor.name, ")   ",
                          "Delta_t:", delta_t, "[s]  (or", delta_t/60, "[min])")

                    # Save results        
                    passes.append((time_of_pass, idx))
                    n_passes += 1

        # Advance time with 1 minute
        instance.advance_time(time_to_advance, 0)


# Plot the constellation at the time of the second pass with LOS of the disaster site
paseos_instances[0].empty_known_actors()
for instance in paseos_instances[1:]:
    paseos_instances[0].add_known_actor(instance.local_actor)
for instance in comms_instances:
    paseos_instances[0].add_known_actor(instance.local_actor)
plotter = paseos.plot(paseos_instances[0], paseos.PlotType.SpacePlot)